# Regression model for rate of bikes per hour

Example of multiple linear regression here: https://nbviewer.org/github/justmarkham/DAT4/blob/master/notebooks/08_linear_regression.ipynb

In [67]:
import pandas as pd
import numpy as np
import pingouin as pg # for repeated measures model
from matplotlib import pyplot as plt
from tqdm import tqdm

In [68]:
import statsmodels.formula.api as smf

In [69]:
agg_counts = pd.read_csv('data/aggregated_data.csv')

In [70]:
agg_counts.columns

Index(['year', 'grouped_counterpoint_id', 'geocoded_city', 'geocoded_province',
       'start_datetime_dayofmonth', 'count_session_label', 'bike_total',
       'pedestrian_total', 'personal_vehicle_total', 'oversized_vehicle_total',
       'num_sessions_grouped', 'unique_counterpoint_ids', 'total_time_seconds',
       'Man_gender_total', 'Woman_gender_total', 'Unsure_age_total',
       'Senior_age_total', 'Adult_age_total', 'Child_age_total',
       'White_race_total', 'South Asian_race_total', 'Indigenous_race_total',
       'East Asian_race_total', 'Black_race_total',
       'Middle Eastern_race_total', 'Latin_race_total', 'Unsure_race_total',
       'Unsure_gender_total'],
      dtype='object')

In [71]:
canbics = pd.read_csv("data/PP_CounterPoints_Cycle_Facilities_FULL_02JAN2025.csv")

In [72]:
canbics['can_bics_facility'] = canbics['can_bics_facility'].str.strip() # remove leading spaces

In [73]:
# Change North Vancouver to Vancouver
agg_counts.loc[agg_counts['geocoded_city'] == 'North Vancouver', 'geocoded_city'] = 'Vancouver'

In [74]:
agg_counts['count_year'] = agg_counts['year']

In [75]:
agg_counts.shape

(1860, 29)

In [76]:
agg_counts = agg_counts.merge(canbics[['grouped_counterpoint_id', 'count_year', 
                                       'can_bics_facility', 'comfort_class', 
                                       'google_street_month', 'google_street_year']].drop_duplicates(),
                 on = ('grouped_counterpoint_id', 'count_year'), how = 'left')

### Spatial access measures

Direct link to data: https://www150.statcan.gc.ca/n1/pub/27-26-0001/2023001/sam-msa-eng.zip

Data landing page here: https://www150.statcan.gc.ca/n1/pub/27-26-0001/272600012023001-eng.htm

In [77]:
# spatial access measures

sam_data = pd.read_csv("data/sam-msa-eng/acs_cycling.csv",
                      encoding='latin-1')

In [78]:
sam_data.columns

Index(['DBUID', 'CSDUID', 'CSDNAME', 'CMAUID', 'CMANAME', 'PRUID', 'PRCODE',
       'acs_idx_hf', 'acs_idx_emp', 'acs_idx_srf', 'acs_idx_psef',
       'acs_idx_ef', 'acs_idx_caf', 'acs_lvl_gs-1', 'acs_lvl_gs-3',
       'acs_lvl_gs-5'],
      dtype='object')

Get dissemination block for each counterpoint location

Direct link to dissemination block boundary file: https://www12.statcan.gc.ca/census-recensement/2021/geo/sip-pis/boundary-limites/files-fichiers/ldb_000a21a_e.zip

To download dissemination block boundary file:
- Go to Statistics Canada website here: https://www12.statcan.gc.ca/census-recensement/2021/geo/sip-pis/boundary-limites/index2021-eng.cfm?year=21
- Under "Type" select "Digital Boundary Files"
- Under "Statistical boundaries" select "Dissemination blocks"
- Under "Format" select "Shapefile"
- Select "Continue"
- Download and extract the zip file on the next page. 

In [79]:
def get_counterpoint_to_db(agg_counts, sam_data):
    import geopandas as gpd

    db_boundaries = gpd.read_file("data/ldb_000a21a_e/ldb_000a21a_e.shp")

    agg_counts_gdf = gpd.GeoDataFrame(
        agg_counts, geometry=gpd.points_from_xy(agg_counts['grouped_counterpoint_lng'],
                                                      agg_counts['grouped_counterpoint_lat']), crs="EPSG:4326"
    )

    unique_counterpoint_gdf = agg_counts_gdf.drop_duplicates(subset = 'grouped_counterpoint_id')
    unique_counterpoint_gdf = unique_counterpoint_gdf.to_crs(db_boundaries.crs)

    counterpoint_to_dbuid = {}
    for i, row in tqdm(unique_counterpoint_gdf.iterrows()):
        dbuid = db_boundaries[db_boundaries['geometry'].contains(row['geometry'])]['DBUID']
        assert len(dbuid) == 1
        counterpoint_to_dbuid[int(row['grouped_counterpoint_id'])] = int(dbuid.iloc[0])

    db_boundaries['DBUID'] = db_boundaries['DBUID'].astype('int')

    db_boundaries_sam = db_boundaries[db_boundaries['DBUID'].isin(sam_data['DBUID'])]

    assert db_boundaries_sam.shape[0] == sam_data.shape[0] # check that all of sam data is in there

    agg_counts['DBUID'] = agg_counts['grouped_counterpoint_id'].map(counterpoint_to_dbuid)
    agg_counts[['grouped_counterpoint_id', 'DBUID']].drop_duplicates().to_csv("data/counterpoint_to_dbuid.csv", index = None)

In [80]:
#counterpoint_id_info = pd.read_csv("counterpoint_id_groupings_edited_names.csv") # this file has counterpoint names edited by hand (more concise)
counterpoint_id_info = pd.read_csv("data/counterpoint_id_groupings.csv") # up to date for 2024

In [81]:
counterpoints = counterpoint_id_info[['grouped_counterpoint_id', 'grouped_counterpoint_lat', 'grouped_counterpoint_lng']].drop_duplicates()

In [82]:
agg_counts = agg_counts.merge(counterpoints, on = 'grouped_counterpoint_id', how = 'left')

In [83]:
try:
    counterpoint_to_dbuid = pd.read_csv("data/counterpoint_to_dbuid.csv")
    agg_counts = agg_counts.merge(counterpoint_to_dbuid, on = 'grouped_counterpoint_id', how = 'left')
    print("loaded counterpoint to DB mapping")
except:
    get_counterpoint_to_db(agg_counts, sam_data)
    counterpoint_to_dbuid = pd.read_csv("data/counterpoint_to_dbuid.csv")
    

loaded counterpoint to DB mapping


In [84]:
agg_counts = agg_counts.merge(sam_data[['DBUID', 'acs_idx_emp']], on = 'DBUID', how = 'left')

In [85]:
assert agg_counts[agg_counts['acs_idx_emp'].isna()].shape[0] == 0

In [86]:
agg_counts.shape

(1860, 37)

Do label encoding for canbics comfort class

In [87]:
agg_counts['can_bics_facility'].unique()

array([nan, 'Cycle track', 'Non-conforming', 'Not classified',
       'Multi-use path', 'Painted bike lane', 'Local street bikeway',
       'Bike path'], dtype=object)

In [88]:
agg_counts[['can_bics_facility', 'comfort_class']].value_counts()

can_bics_facility     comfort_class
Multi-use path        Medium           671
Cycle track           High             286
Painted bike lane     Low              116
Local street bikeway  High             109
Bike path             High              62
Name: count, dtype: int64

In [89]:
comfort_class_dict = {'Low':1, 'Medium':2, 'High':3}

In [90]:
# should we try including non-conforming as well?
canbics_facility_dict = {'Painted bike lane':1, 'Not classified':0, 'Cycle track':3,
       'Multi-use path':2, 'Local street bikeway':3,
       'Bike path':3}

In [91]:
agg_counts['comfort_class_n'] = agg_counts['comfort_class'].map(comfort_class_dict)
agg_counts['can_bics_n'] = agg_counts['can_bics_facility'].map(canbics_facility_dict)

Label encoding for weekday vs weekend

In [92]:
agg_counts['weekday_weekend'] = agg_counts['count_session_label'].str.split('_', expand = True)[0]

In [93]:
weekday_weekend_dict = {'weekday':1, 'weekend':-1}

In [94]:
agg_counts['weekday_weekend_n'] = agg_counts['weekday_weekend'].map(weekday_weekend_dict)

## Regression model


Repeated measures. The problem here is that we actually only have a small fraction of the data at repeated sites. 

Repeated measures is basically a special case of mixed-effects modelling / multilevel modelling. See Bakdash2017.

Also, in almost every case, if we use Can-BICS or accessibility as the independent variable, it is not changing at all for repeated sites. In this way, the measurements can be thought of as independent. The only variable for which repeated measures make sense is year, and maybe weekday_weekend if the subjects are sites and not site-session combinations?

We detected very few cases of possible infrastructure changes at count sites. Because of this, when modelling the relationship of counts to infrastructure and accessibility, we treat each observation as independent, even observations at different time points at the same site, because the independent variables are not changing within sites in the vast majority of observations. 

To assess the effect of time on results, we use a linear mixed effects model

In [95]:
agg_counts.shape

(1860, 41)

In [96]:
agg_counts[agg_counts['count_session_label'] == 'not_assigned'].shape

(54, 41)

Remove unassigned count sessions

In [97]:
agg_counts = agg_counts[agg_counts['count_session_label'] != 'not_assigned']

In [98]:
agg_counts.shape

(1806, 41)

In [99]:
# make a column for the combination of location and session
agg_counts['location_session'] = agg_counts['grouped_counterpoint_id'].astype('int').astype('str') + "_" + agg_counts['count_session_label']

For some count locations and count time windows, there were multiple count sessions during the same time window on different days in the same count year. Because our analysis for trends over time compares matched sessions at the same location and time window over multiple years, we first averaged the results from repeated count sessions in the same year. Of 1858 count sessions in the data, 49 sessions were counted more than once in the same year.

We first removed duplicated count sessions that were less than 30 minutes in length. In all cases, there was at least one other count in the same time window that was at least 45 minutes long. 12 short count sessions were dropped. For the remaining 37 cases, we averaged the bikes per hour across counts at the same time window, year, and count location, giving 1802 unique count sessions.

In [100]:
duplicated_sessions = agg_counts.groupby(['year', 'location_session'])['grouped_counterpoint_id'].count().reset_index().sort_values(by = 
                                                                            'grouped_counterpoint_id', ascending = False)

In [101]:
duplicated_sessions = duplicated_sessions[duplicated_sessions['grouped_counterpoint_id']  > 1]

In [102]:
duplicated_sessions.shape

(47, 3)

In [103]:
agg_counts['location_session_year'] = agg_counts['year'].astype('str') + '_' + agg_counts['location_session']
duplicated_sessions['location_session_year'] = duplicated_sessions['year'].astype('str') + '_' + duplicated_sessions['location_session']

In [104]:
duplicated_sessions_full = agg_counts[agg_counts['location_session_year'].isin(duplicated_sessions['location_session_year'])]


In [105]:
# duplicated sessions less than 30 minutes in length - make sure we don't remove both sessions by doing this. 
# This checks that there is only one of each session combo in this list
assert np.all(duplicated_sessions_full[duplicated_sessions_full['total_time_seconds'] < 30*60]['location_session_year'].value_counts() == 1)

In [106]:
short_sessions = duplicated_sessions_full[duplicated_sessions_full['total_time_seconds'] < 30*60]['location_session_year'].value_counts().index

In [107]:
short_sessions.shape

(11,)

In [108]:
# check pairs of times - is at least one of the counts much longer?
duplicated_sessions_full[duplicated_sessions_full['location_session_year'].isin(short_sessions)][['location_session_year',
                                                                                                  'total_time_seconds']].sort_values(
    by = 'location_session_year').groupby('location_session_year')['total_time_seconds'].agg(['max', 'min']).reset_index()

,location_session_year,max,min
0,2021_5180_weekday_afternoon,7200,420
1,2021_5314_weekday_afternoon,7200,960
2,2021_5391_weekday_afternoon,7080,1680
3,2021_5452_weekday_afternoon,3480,420
4,2021_5569_weekday_late_afternoon,7200,1380
5,2022_5389_weekday_afternoon,7200,300
6,2022_5403_weekday_lunch,7260,1620
7,2022_6409_weekday_lunch,7140,1260
8,2022_6430_weekend_afternoon,5040,600
9,2023_6497_weekday_afternoon,2765,911


#### Remove duplicated counts less than 30 minutes and average remaining duplicated counts

In [111]:
agg_counts['bikes_per_hour'] = (agg_counts['bike_total'] / agg_counts['total_time_seconds'])*3600

In [112]:
# remove duplicated counts that are less than 30 minutes
agg_counts_mean = agg_counts[~((agg_counts['location_session_year'].isin(short_sessions))
           & (agg_counts['total_time_seconds'] < 30*60))]

In [113]:
bikes_per_hour_means = agg_counts_mean.groupby('location_session_year')['bikes_per_hour'].mean().reset_index()

In [114]:
bikes_per_hour_means_dict = dict(zip(bikes_per_hour_means['location_session_year'], bikes_per_hour_means['bikes_per_hour']))

In [115]:
agg_counts_mean['bikes_per_hour_mean'] = agg_counts_mean['location_session_year'].map(bikes_per_hour_means_dict)

/tmp/ipykernel_6081/2921511946.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  agg_counts_mean['bikes_per_hour_mean'] = agg_counts_mean['location_session_year'].map(bikes_per_hour_means_dict)


In [116]:
# drop duplicate sessions
agg_counts_mean = agg_counts_mean.sort_values(by = 'total_time_seconds', 
                                              ascending = False).drop_duplicates(subset = ['location_session_year', 
                                                                                           'bikes_per_hour_mean'], keep = 'first')

In [117]:
# remane columns
agg_counts_mean['bikes_per_hour_orig'] = agg_counts_mean['bikes_per_hour']
agg_counts_mean['bikes_per_hour'] = agg_counts_mean['bikes_per_hour_mean']

In [118]:
agg_counts_mean[agg_counts_mean['can_bics_facility'] ==  'Non-conforming'].shape

(135, 46)

In [119]:
agg_counts_mean[agg_counts_mean['can_bics_facility'].isna()].shape

(139, 46)

In [120]:
multi_year_sessions = (agg_counts['location_session'].value_counts())[agg_counts['location_session'].value_counts() > 1].index

In [121]:
agg_counts_multiyear = agg_counts_mean[agg_counts_mean['location_session'].isin(multi_year_sessions)]

In [122]:
agg_counts_multiyear.shape

(1111, 46)

In [125]:
1752-135-139

1478

In [126]:
agg_counts_mean.shape

(1752, 46)

In [182]:
year_counts_grouped = agg_counts_mean.groupby(['grouped_counterpoint_id', 'count_session_label'])['year'].count().reset_index()

In [187]:
# sites counted in 2 or more years during same time window
year_counts_grouped[year_counts_grouped['year'] > 1].shape

(443, 3)

In [189]:
year_counts_grouped[year_counts_grouped['year'] == 1].shape

(656, 3)

#### Random intercept model for site & time window combination

In [127]:
data_subset = agg_counts_mean.dropna(subset = ['can_bics_n', 'acs_idx_emp'])

In [128]:
data_subset.shape

(1478, 46)

In [129]:
md = smf.mixedlm(formula="bikes_per_hour ~ year + can_bics_n + acs_idx_emp", 
                 data=data_subset, groups=data_subset['location_session'])

mdf_all = md.fit()

print(mdf_all.summary())

            Mixed Linear Model Regression Results
Model:              MixedLM Dependent Variable: bikes_per_hour
No. Observations:   1478    Method:             REML          
No. Groups:         885     Scale:              3939.4439     
Min. group size:    1       Log-Likelihood:     -8658.8552    
Max. group size:    4       Converged:          Yes           
Mean group size:    1.7                                       
--------------------------------------------------------------
              Coef.   Std.Err.   z    P>|z|   [0.025   0.975] 
--------------------------------------------------------------
Intercept   -1242.585 3516.269 -0.353 0.724 -8134.345 5649.176
year            0.604    1.739  0.348 0.728    -2.804    4.013
can_bics_n     16.239    2.748  5.909 0.000    10.853   21.626
acs_idx_emp   345.095   16.931 20.382 0.000   311.911  378.280
Group Var    4567.941    7.875                                



In [130]:
# year alone
md = smf.mixedlm(formula="bikes_per_hour ~ year", 
                 data=agg_counts_multiyear, groups=agg_counts_multiyear['location_session'])

mdf = md.fit()

print(mdf.summary())

           Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: bikes_per_hour
No. Observations: 1111    Method:             REML          
No. Groups:       458     Scale:              3812.7980     
Min. group size:  1       Log-Likelihood:     -6590.2367    
Max. group size:  4       Converged:          Yes           
Mean group size:  2.4                                       
------------------------------------------------------------
            Coef.   Std.Err.   z    P>|z|   [0.025   0.975] 
------------------------------------------------------------
Intercept -2249.737 3739.338 -0.602 0.547 -9578.705 5079.232
year          1.147    1.849  0.620 0.535    -2.477    4.771
Group Var  9385.905   15.375                                



### Regression coefficients in terms of interquartile range

Also known as "robust standardized coefficient", deals better with the presence of outliers or non-standard data distributions and allows better comparability across independent variables. https://nielsen.sites.oasis.unc.edu/soci709/m5/m5std.html

Formula: unstardized regression coefficient times (interquartile range X / interquartile range Y)

Or, can just use the interquartile range of X to keep Y in its own units

In [131]:
from scipy import stats

In [132]:
stats.iqr(data_subset['bikes_per_hour'])

np.float64(56.13697409561456)

In [133]:
np.quantile(data_subset['bikes_per_hour'], 0.25)

np.float64(12.36302590438544)

In [134]:
np.quantile(data_subset['bikes_per_hour'], 0.75)

np.float64(68.5)

In [135]:
# interquartile range is the 75th percentile of data minus the 25th percentile of data
np.quantile(data_subset['bikes_per_hour'], 0.75) - np.quantile(data_subset['bikes_per_hour'], 0.25)

np.float64(56.13697409561456)

In [136]:
md = smf.mixedlm(formula="bikes_per_hour ~ year + can_bics_n + acs_idx_emp", 
                 data=data_subset, groups=data_subset['location_session'])

mdf_all = md.fit()

print(mdf_all.summary())

            Mixed Linear Model Regression Results
Model:              MixedLM Dependent Variable: bikes_per_hour
No. Observations:   1478    Method:             REML          
No. Groups:         885     Scale:              3939.4439     
Min. group size:    1       Log-Likelihood:     -8658.8552    
Max. group size:    4       Converged:          Yes           
Mean group size:    1.7                                       
--------------------------------------------------------------
              Coef.   Std.Err.   z    P>|z|   [0.025   0.975] 
--------------------------------------------------------------
Intercept   -1242.585 3516.269 -0.353 0.724 -8134.345 5649.176
year            0.604    1.739  0.348 0.728    -2.804    4.013
can_bics_n     16.239    2.748  5.909 0.000    10.853   21.626
acs_idx_emp   345.095   16.931 20.382 0.000   311.911  378.280
Group Var    4567.941    7.875                                



In [137]:
mdf_all.params

Intercept     -1242.584574
year              0.604398
can_bics_n       16.239105
acs_idx_emp     345.095179
Group Var         1.159539
dtype: float64

In [138]:
# calculate interquartile range for each independent variable
iqr_bikes_per_hour = stats.iqr(data_subset['bikes_per_hour'])
iqr_canbics = stats.iqr(data_subset['can_bics_n'])
iqr_sam = stats.iqr(data_subset['acs_idx_emp'])
iqr_year = stats.iqr(data_subset['year'])

In [139]:
# rescaled year coefficient
mdf_all.params.year * (iqr_year / iqr_bikes_per_hour)

np.float64(0.021532975813217375)

In [140]:
mdf_all.params.year * (iqr_year)

np.float64(1.2087961054280787)

In [141]:
mdf_all.params.can_bics_n * (iqr_canbics / iqr_bikes_per_hour)

np.float64(0.5785529131725556)

In [142]:
mdf_all.params.can_bics_n * (iqr_canbics)

np.float64(32.478209899710095)

In [143]:
mdf_all.params.acs_idx_emp * (iqr_sam / iqr_bikes_per_hour)

np.float64(0.9201859971654065)

In [144]:
mdf_all.params.acs_idx_emp * (iqr_sam)

np.float64(51.65645748602168)

Moving across the interquartile range causes an XX inrease in bikes per hour.

In [166]:
mdf_all.pvalues

Intercept      7.238024e-01
year           7.281599e-01
can_bics_n     3.446262e-09
acs_idx_emp    2.403001e-92
Group Var      2.419459e-20
dtype: float64

In [146]:
4.821424e-10 < 10**(-9)

True

In [147]:
6.577463e-01

0.6577463

In [148]:
# make list to multiply confidence interval
coefs_iqr = [iqr_year, 
             iqr_canbics,
             iqr_sam]

In [149]:
mdf_all.conf_int()

,0,1
Intercept,-8134.345172,5649.176023
year,-2.803789,4.012585
can_bics_n,10.852545,21.625665
acs_idx_emp,311.910727,378.279631
Group Var,0.913634,1.405445


In [150]:
# left side of confidence interval
mdf_all.conf_int().iloc[1:-1,:][0] * np.array(coefs_iqr)

year           -5.607578
can_bics_n     21.705090
acs_idx_emp    46.689158
Name: 0, dtype: float64

In [151]:
# right side of confidence interval
mdf_all.conf_int().iloc[1:-1,:][1] * np.array(coefs_iqr)

year            8.025170
can_bics_n     43.251330
acs_idx_emp    56.623757
Name: 1, dtype: float64

Construct the full table here for clarity

In [170]:
reg_df = pd.DataFrame()

reg_df['variable'] = ['year', 'can-bics', 'SAM']
reg_df['coefficient'] = [round(mdf_all.params.year * (iqr_year),2),
                         round(mdf_all.params.can_bics_n * (iqr_canbics),2),
                         round(mdf_all.params.acs_idx_emp * (iqr_sam), 2)]
reg_df['pvalue'] =  list(mdf_all.pvalues[1:-1])
reg_df['conf_left'] = list(mdf_all.conf_int().iloc[1:-1,:][0] * np.array(coefs_iqr))
reg_df['conf_right'] = list(mdf_all.conf_int().iloc[1:-1,:][1] * np.array(coefs_iqr))

In [171]:
reg_df

,variable,coefficient,pvalue,conf_left,conf_right
0,year,1.21,7.281599e-01,-5.607578,8.025170
1,can-bics,32.48,3.446262e-09,21.705090,43.251330
2,SAM,51.66,2.403001e-92,46.689158,56.623757
